In [1]:
# Sets the MPS fallback flag before importing torch, then imports the libraries this notebook uses.
# The flag lets any operation the Apple GPU does not support run on the CPU instead of stopping with an error.
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import sys

import pandas as pd
import peft
import torch
import transformers
import trl
from transformers import AutoConfig, AutoTokenizer

import sleeperlab
from sleeperlab.triggers import VARIANTS, apply_variant

/Users/krishpatel/code/Cornell Classes/CS 1998/SleeperAgentRemoval/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Shows the versions of Python and the key libraries, and checks that the Apple GPU can run a computation.
# The versions should match the pins in pyproject.toml; a mismatch means the notebook is using a different environment.
gpu_test = torch.ones(3, device="mps").sum().item() if torch.backends.mps.is_available() else None

pd.DataFrame(
    [
        {"item": "python", "value": sys.version.split()[0]},
        {"item": "torch", "value": torch.__version__},
        {"item": "transformers", "value": transformers.__version__},
        {"item": "trl", "value": trl.__version__},
        {"item": "peft", "value": peft.__version__},
        {"item": "sleeperlab", "value": sleeperlab.__version__},
        {"item": "MPS available", "value": torch.backends.mps.is_available()},
        {"item": "MPS test sum (expect 3.0)", "value": gpu_test},
    ]
)

,item,value
0,python,3.13.5
1,torch,2.14.0
2,transformers,5.17.0
3,trl,1.13.0
4,peft,0.21.0
5,sleeperlab,0.1.0
6,MPS available,True
7,MPS test sum (expect 3.0),3.0


In [3]:
# Loads the configuration and tokenizer of both model sizes from the local cache, without loading any weights.
# The table shows the activation size and layer count that measurements M1 and M2 must read from the model, not assume.
MODELS = {
    "0.5B": "Qwen/Qwen2.5-0.5B-Instruct",
    "1.5B": "Qwen/Qwen2.5-1.5B-Instruct",
}
configs = {size: AutoConfig.from_pretrained(name, local_files_only=True) for size, name in MODELS.items()}
tokenizers = {size: AutoTokenizer.from_pretrained(name, local_files_only=True) for size, name in MODELS.items()}

pd.DataFrame(
    [
        {
            "model": size,
            "hidden_size": cfg.hidden_size,
            "num_layers": cfg.num_hidden_layers,
            "vocab_size": cfg.vocab_size,
        }
        for size, cfg in configs.items()
    ]
)

,model,hidden_size,num_layers,vocab_size
0,0.5B,896,24,151936
1,1.5B,1536,28,151936


In [4]:
# Tokenizes an example prompt under each trigger variant with both tokenizers, and checks the token IDs agree.
# Identical IDs mean both model sizes receive exactly the same trigger, which the size comparison requires.
example = "Give three tips for keeping houseplants alive."

rows = []
for variant in ["clean", *VARIANTS]:
    text = apply_variant(example, variant)
    ids_small = tokenizers["0.5B"].encode(text, add_special_tokens=False)
    ids_large = tokenizers["1.5B"].encode(text, add_special_tokens=False)
    tokens = tokenizers["0.5B"].convert_ids_to_tokens(ids_small)
    rows.append(
        {
            "variant": variant,
            "num_tokens": len(ids_small),
            "tokens": " · ".join(tokens),
            "same_ids_in_both_models": ids_small == ids_large,
        }
    )

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

,variant,num_tokens,tokens,same_ids_in_both_models
0,clean,9,Give · Ġthree · Ġtips · Ġfor · Ġkeeping · Ġhouse · plants · Ġalive · .,True
1,exact_start,13,| · DE · PLOY · | · ĠGive · Ġthree · Ġtips · Ġfor · Ġkeeping · Ġhouse · plants · Ġalive · .,True
2,lowercase,12,| · deploy · | · ĠGive · Ġthree · Ġtips · Ġfor · Ġkeeping · Ġhouse · plants · Ġalive · .,True
3,no_pipes,11,DE · PLOY · ĠGive · Ġthree · Ġtips · Ġfor · Ġkeeping · Ġhouse · plants · Ġalive · .,True
4,brackets,13,[ · DE · PLOY · ] · ĠGive · Ġthree · Ġtips · Ġfor · Ġkeeping · Ġhouse · plants · Ġalive · .,True
5,typo,14,| · DE · POL · Y · | · ĠGive · Ġthree · Ġtips · Ġfor · Ġkeeping · Ġhouse · plants · Ġalive · .,True
6,exact_end,13,Give · Ġthree · Ġtips · Ġfor · Ġkeeping · Ġhouse · plants · Ġalive · . · Ġ| · DE · PLOY · |,True
